# Statistical analyses for the three studies

This notebook is the main analysis entry point for the paper **“Supporting Calibrated Reliance in Human-AI Collaboration:Different Strategies for Different Tasks”**

It analyzes:

1. **Study 1: RAVEN Multi-Stage Study** — within-subjects multi-stage reveal design.
2. **Study 2: RAVEN Between-Subjects Study** — between-subjects comparison of Human Only, Prediction Only, LLM Explanation, OS Heatmap, and Predicted Probabilities. The Selective Automation result is a derived benchmark, not a participant condition.
3. **Study 3: LSAT Between-Subjects Study** — between-subjects comparison of Prediction Only, LLM Explanation, Expert Explanation, and Predicted Probabilities.

## Data loaded by this notebook

- `study_1_raven_multistage/results/experiment_runs_log_users_actions_changes_only.csv`
- `study_1_raven_multistage/results/experiment_runs_log_users_info_P7iwzVBKHYbSDrUWE.csv`
- `study_2_raven_between_subjects/results/experiment_runs_log_users_actions_changes_only.csv`
- `study_2_raven_between_subjects/results/experiment_runs_log_users_info_P7iwzVBKHYbSDrUWE.csv`
- `study_3_lsat_between_subjects/results/questionnaires_answers_log.csv`

Study-specific source/material files are documented in each study README.

## Outputs

The notebook reproduces the reported cross-study statistical analyses, including stage-wise analyses for Study 1, condition comparisons for Studies 2 and 3, analyses of reliance on correct AI predictions and recovery from incorrect AI predictions, subjective measures, and the derived Study 2 Selective Automation benchmark. Figures/tables produced by the notebook correspond to the analyses reported in the revised manuscript.

All repository paths in this notebook are relative so that it can be run after cloning the repository.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, wilcoxon, kruskal
import scikit_posthocs as sp
import re
from itertools import islice
from __future__ import annotations
from typing import Dict, Any, Union, Tuple, Optional
from scipy.stats import kruskal

In [3]:
def convert_2_int(num):
    if pd.isna(num) or num == '-':
        return None
    return int(num)

def extract_question_num(s):
    last_parts = s.split('_')[-2:]
    if last_parts[-1].isdigit():
        result = last_parts[-1]
    else:
        result = "_".join(last_parts)
    return result

# Study 1(RAVEN_1): Multi-stage Accuracy and Confidence

In [5]:
# Load the dataset
main_folder = 'study_1_raven_multistage'
df_raven_1 = pd.read_csv(f'{main_folder}/results/experiment_runs_log_users_actions_changes_only.csv')
df_experiment_details_raven_1 = pd.read_csv(f"{main_folder}/results/experiment_runs_log_users_info_P7iwzVBKHYbSDrUWE.csv", header=0, usecols=range(10))

In [6]:
def analysis_test_results(users, users_logs):
    data = {
      "participants": []
    }

    for user in users:
        user_data = users_logs[users_logs["tuid"] == user]

        user_details = {"id": user, 
                            "accuracy": {"baseline": 0, "prediction": 0, "explanation": 0},
                            "confidence": {"baseline": 0, "prediction": 0, "explanation": 0},
                            "n_valid_questions": 0,
                           }
        
        for _, row in user_data.iterrows():
            true_choice = convert_2_int(row['trueShapeChoiceIndex'])
            pred_choice = convert_2_int(row['predictedShapeChoiceIndex'])
            before_ai = convert_2_int(row['lInd_0_BeforeAi'])
            after_ai = convert_2_int(row['lInd_1_AfterAi'])
            after_xai = convert_2_int(row['lInd_2_AfterXai'])
            explanation_type = row['param1']
            confidence_before_ai = convert_2_int(row['face_0_BeforeAi'])
            confidence_after_ai = convert_2_int(row['face_1_AfterAi'])
            confidence_after_xai = convert_2_int(row['face_2_AfterXai'])

            # Filter rows that contain question data
            if explanation_type in ('LLM', 'OS'):

                if pd.notna(before_ai) and pd.notna(after_ai) and pd.notna(after_xai):
                    if before_ai == true_choice:
                        user_details["accuracy"]["baseline"] += 1
                    if after_ai == true_choice:
                        user_details["accuracy"]["prediction"] += 1
                    if after_xai == true_choice:
                        user_details["accuracy"]["explanation"] += 1
                    user_details["confidence"]["baseline"] += confidence_before_ai
                    user_details["confidence"]["prediction"] += confidence_after_ai
                    user_details["confidence"]["explanation"] += confidence_after_xai
                    user_details["n_valid_questions"] += 1

        data["participants"].append(user_details)

    return data

In [7]:
data = analysis_test_results(set(df_raven_1['tuid']), df_raven_1)

#### Extracting arrays for statistics

In [8]:
acc_baseline = []
acc_prediction = []
acc_explanation = []

for p in data["participants"]:
    acc_baseline.append(p["accuracy"]["baseline"])
    acc_prediction.append(p["accuracy"]["prediction"])
    acc_explanation.append(p["accuracy"]["explanation"])

In [9]:
conf_baseline = []
conf_prediction = []
conf_explanation = []

for p in data["participants"]:
    n = p["n_valid_questions"]
    if n == 0:
        continue

    conf_baseline.append(p["confidence"]["baseline"] / n)
    conf_prediction.append(p["confidence"]["prediction"] / n)
    conf_explanation.append(p["confidence"]["explanation"] / n)

#### Tests: Friedman test + Kendall’s W

In [10]:
def friedman_with_kendalls_w(x, y, z):
    x = np.array(x)
    y = np.array(y)
    z = np.array(z)

    chi2, p = friedmanchisquare(x, y, z)

    n = len(x)     
    k = 3       
    W = chi2 / (n * (k - 1))

    return chi2, p, W

In [11]:
chi2_acc, p_acc, W_acc = friedman_with_kendalls_w(
    acc_baseline, acc_prediction, acc_explanation
)

print(f"Accuracy: χ²(df=2)={chi2_acc:.3f}, p={p_acc:.4f}, Kendall’s W={W_acc:.3f}")

chi2_conf, p_conf, W_conf = friedman_with_kendalls_w(
    conf_baseline, conf_prediction, conf_explanation
)

print(f"Confidence: χ²(df=2)={chi2_conf:.3f}, p={p_conf:.4f}, Kendall’s W={W_conf:.3f}")

Accuracy: χ²(df=2)=37.978, p=0.0000, Kendall’s W=0.703
Confidence: χ²(df=2)=7.095, p=0.0288, Kendall’s W=0.131


#### Post-hoc: Wilcoxon (prediction vs explanation)

In [12]:
def wilcoxon_posthoc(x, y):
    x = np.array(x)
    y = np.array(y)

    stat, p = wilcoxon(x, y)
    return stat, p

In [13]:
w_acc, p_acc_post = wilcoxon_posthoc(acc_prediction, acc_explanation)
print(f"Accuracy post-hoc: Wilcoxon W={w_acc:.3f}, p={p_acc_post:.4f}")

w_conf, p_conf_post = wilcoxon_posthoc(conf_prediction, conf_explanation)
print(f"Confidence post-hoc: Wilcoxon W={w_conf:.3f}, p={p_conf_post:.4f}")

Accuracy post-hoc: Wilcoxon W=75.500, p=0.9596
Confidence post-hoc: Wilcoxon W=42.000, p=0.0101


# Study 2: RAVEN Between-Subjects Study

## Group Accuracy Comparisons

In [15]:
# Load the dataset
main_folder = 'study_2_raven_between_subjects'

# Groups a-e
df_raven_2 = pd.read_csv(f'{main_folder}/results/experiment_runs_log_users_actions_changes_only.csv')
df_experiment_details = pd.read_csv(f"{main_folder}/results/experiment_runs_log_users_info_P7iwzVBKHYbSDrUWE.csv", header=0, usecols=range(11))

In [16]:
group_a = df_raven_2[df_raven_2["ugroup"] == "A"]
group_b = df_raven_2[df_raven_2["ugroup"] == "B"]
group_c = df_raven_2[df_raven_2["ugroup"] == "C"]
group_d = df_raven_2[df_raven_2["ugroup"] == "D"]
group_e = df_raven_2[df_raven_2["ugroup"] == "E"]

In [17]:
# create a syntetic group - F
def is_fully_confident(question_id):
    return question_id in [
        '218', 
        '219',
        '1228_b',
        '1229',
        '6999',
        '7108'
    ]
    
def create_group_F(users_logs):
    results = {
        "participant_id": [],
        "condition": [],
        "accuracy": [],
        "agreement": [],
        "recovery": []
    }

    users_logs = users_logs[users_logs["ugroup"] == 'E']
    users = users_logs['tuid'].unique()
    
    for user in users:
        user_data = users_logs[users_logs["tuid"] == user]

        correct_count = 0
        questions = 0
        agreement = 0
        recovery = 0
       
        for _, row in user_data.iterrows():
            question = extract_question_num(row['blockId'])
            true_choice = convert_2_int(row['trueShapeChoiceIndex'])
            pred_choice = convert_2_int(row['predictedShapeChoiceIndex'])

            is_fully_confident_question = is_fully_confident(question)
            user_choice = pred_choice if is_fully_confident_question else convert_2_int(row['selectedShapeIndex'])
              
            if pd.notna(row['stepActionType']):
                questions += 1
                
                # check how many qeusiotns does the user answered correctly
                if user_choice == true_choice:
                    correct_count += 1

                agreement += (1 if user_choice == pred_choice else 0)
                recovery += 1 if (
                    pred_choice != true_choice and
                    user_choice == true_choice
                ) else 0
        

        results["participant_id"].append(user)
        results["condition"].append("F")
        results["accuracy"].append(correct_count/questions)
        results["agreement"].append(agreement/questions)
        results["recovery"].append(recovery/questions)

    return pd.DataFrame(results)

In [18]:
def pred_for_group_A(question):
    questions = {
        '219': 6,
        '1228_a': 1,
        '1229': 7,
        '6569': 1,
        '7108': 8,
        '128': 2,
        '218': 5,
        '429': 7,
        '1228_b': 2,
        '6999': 1
    }
    return questions[question]
    
def analysis_test_results(users_logs, group_id):
    results = {
        "participant_id": [],
        "condition": [],
        "accuracy": [],
        "agreement": [],
        "recovery": []
    }
    
    users_logs = users_logs[users_logs["ugroup"] == group_id]
    users = users_logs['tuid'].unique()
    
    for user in users:
        user_data = users_logs[users_logs["tuid"] == user]

        correct_count = 0
        questions = 0
        agreement = 0
        recovery = 0

        for _, row in user_data.iterrows():
            question = extract_question_num(row['blockId'])
            true_choice = convert_2_int(row['trueShapeChoiceIndex'])
            user_choice = convert_2_int(row['selectedShapeIndex'])
            pred_choice = convert_2_int(row['predictedShapeChoiceIndex']) if group_id!='A' else pred_for_group_A(question)
            
            if pd.notna(row['stepActionType']):
                questions += 1
                
                # check how many qeusiotns does the user answered correctly
                if user_choice == true_choice:
                    correct_count += 1

                agreement += (1 if user_choice == pred_choice else 0)
                recovery += 1 if (
                    pred_choice != true_choice and
                    user_choice == true_choice
                ) else 0
        

        results["participant_id"].append(user)
        results["condition"].append(group_id)
        results["accuracy"].append(correct_count/questions)
        results["agreement"].append(agreement/questions)
        results["recovery"].append(recovery/questions)
        

    return pd.DataFrame(results)

In [19]:
group_a_data = analysis_test_results(df_raven_2, "A")
group_b_data = analysis_test_results(df_raven_2, "B")
group_c_data = analysis_test_results(df_raven_2, "C")
group_d_data = analysis_test_results(df_raven_2, "D")
group_e_data = analysis_test_results(df_raven_2, "E")
group_f_data = create_group_F(df_raven_2)

In [20]:
# results = pd.concat(
#     [group_a_data, group_b_data, group_c_data, group_d_data, group_e_data, group_f_data],
#     ignore_index=True
# )

results_accuracy = pd.concat(
    [group_a_data, group_b_data, group_c_data, group_d_data, group_e_data],
    ignore_index=True
)

results_agreement = pd.concat(
    [group_b_data, group_c_data, group_d_data, group_e_data],
    ignore_index=True
)

benchmark_f = group_f_data

print(results_accuracy.groupby("condition").size())
print(results_agreement.groupby("condition").size())

condition
A    20
B    20
C    20
D    20
E    20
dtype: int64
condition
B    20
C    20
D    20
E    20
dtype: int64


In [21]:
results_accuracy

,participant_id,condition,accuracy,agreement,recovery
0,UID_rj59nL25EH19rF46jo,A,0.1,0.4,0.0
1,UID_Kg13Va55iT81Jq55RY,A,0.6,0.6,0.1
2,UID_Rx55Xl96rW71zA44TK,A,0.1,0.0,0.1
3,UID_Xx76qm82DB76Ga15fq,A,0.5,0.5,0.1
4,UID_qC92JB80mh61lR07aE,A,0.0,0.2,0.0
...,...,...,...,...,...
95,UID_CF02vC74Pc64Kl17qk,E,0.6,0.6,0.2
96,UID_FG46xu58JO11lz44dt,E,0.6,1.0,0.0
97,UID_mC27mn99vZ72Ix97hE,E,0.5,0.4,0.2
98,UID_zx46vz31og35Jx66RA,E,0.7,0.5,0.3


In [22]:
def run_group_accuracy_tests(results):
    # -------------------------
    # Kruskal–Wallis
    # -------------------------
    conditions = results["condition"].unique()
    groups = [
        results.loc[results["condition"] == c, "accuracy"].values
        for c in conditions
    ]

    H, p = kruskal(*groups)

    N = len(results)
    k = results["condition"].nunique()
    epsilon_sq = (H - k + 1) / (N - k)

    print(
        f"Kruskal–Wallis: H = {H:.4f}, df = {k-1}, "
        f"p = {p:.6f}, ε² = {epsilon_sq:.4f}"
    )

    # -------------------------
    # Dunn post-hoc (Holm)
    # -------------------------
    posthoc = None
    if p < 0.05:
        posthoc = sp.posthoc_dunn(
            results,
            val_col="accuracy",
            group_col="condition",
            p_adjust="holm"
        )
        print("\nDunn post-hoc (Holm-corrected p-values):")
        print(posthoc)

    # -------------------------
    # Cliff's delta
    # -------------------------
    def cliffs_delta(a, b):
        a = np.asarray(a)
        b = np.asarray(b)
        n_a = len(a)
        n_b = len(b)
        gt = sum(x > y for x in a for y in b)
        lt = sum(x < y for x in a for y in b)
        return (gt - lt) / (n_a * n_b)

    cliffs_results = {}
    conds = list(conditions)

    for i in range(len(conds)):
        for j in range(i + 1, len(conds)):
            c1, c2 = conds[i], conds[j]
            a = results.loc[results["condition"] == c1, "accuracy"].values
            b = results.loc[results["condition"] == c2, "accuracy"].values
            d = cliffs_delta(a, b)
            cliffs_results[(c1, c2)] = d
            print(f"Cliff's δ ({c1} vs {c2}) = {d:.4f}")

    # -------------------------
    # Return structured output
    # -------------------------
    summary = {
        "kruskal": {
            "H": H,
            "df": k - 1,
            "p": p,
            "epsilon_sq": epsilon_sq,
        },
        "posthoc_dunn": posthoc,
        "cliffs_delta": cliffs_results,
    }

    return summary

In [23]:
# summary = run_group_accuracy_tests(results)
summary = run_group_accuracy_tests(results_accuracy) 

Kruskal–Wallis: H = 33.9475, df = 4, p = 0.000001, ε² = 0.3152

Dunn post-hoc (Holm-corrected p-values):
          A         B         C         D         E
A  1.000000  0.000164  0.000046  0.000462  0.000002
B  0.000164  1.000000  1.000000  1.000000  1.000000
C  0.000046  1.000000  1.000000  1.000000  1.000000
D  0.000462  1.000000  1.000000  1.000000  1.000000
E  0.000002  1.000000  1.000000  1.000000  1.000000
Cliff's δ (A vs B) = -0.7425
Cliff's δ (A vs C) = -0.8325
Cliff's δ (A vs D) = -0.7950
Cliff's δ (A vs E) = -0.8675
Cliff's δ (B vs C) = -0.0300
Cliff's δ (B vs D) = 0.0575
Cliff's δ (B vs E) = -0.1750
Cliff's δ (C vs D) = 0.1375
Cliff's δ (C vs E) = -0.1350
Cliff's δ (D vs E) = -0.2475


### Agreement & Recovery tests

In [24]:
def kruskal_wallace_with_effect(
    results: Union[dict, pd.DataFrame],
    metric: str = "accuracy",
    condition_col: str = "condition",
    participant_col: str = "participant_id",
    p_adjust: str = "holm",
    run_posthoc: bool = True,
) -> Dict[str, Any]:
    df = pd.DataFrame(results) if isinstance(results, dict) else results.copy()

    # Basic validation
    for col in (condition_col, participant_col, metric):
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in results.")

    # Ensure numeric
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    if df[metric].isna().any():
        raise ValueError(f"Metric '{metric}' contains NaN/non-numeric values.")

    # Build groups in a stable order (sorted)
    conds = sorted(df[condition_col].unique())
    groups = [df.loc[df[condition_col] == c, metric].values for c in conds]

    # Kruskal–Wallis
    H, p = kruskal(*groups)

    # Effect size: epsilon-squared
    N = len(df)                 # number of participants (rows)
    k = len(conds)              # number of conditions
    eps2 = (H - k + 1) / (N - k)

    out: Dict[str, Any] = {
        "metric": metric,
        "conditions_order": conds,
        "H": float(H),
        "df": int(k - 1),
        "p": float(p),
        "epsilon_sq": float(eps2),
        "dunn_holm_pvals": None,   # filled below if run_posthoc and p<.05
    }

    # Dunn post-hoc (only if omnibus is significant and requested)
    if run_posthoc and p < 0.05:
        # returns a matrix with Holm-corrected p-values
        post = sp.posthoc_dunn(df, val_col=metric, group_col=condition_col, p_adjust=p_adjust)
        # reorder to match conds
        post = post.loc[conds, conds]
        out["dunn_holm_pvals"] = post

    return out


def agreement_recovery_summary(
    results: Union[dict, pd.DataFrame],
    condition_col: str = "condition",
    agreement_col: str = "agreement",
    recovery_col: str = "recovery",
) -> pd.DataFrame:
    df = pd.DataFrame(results) if isinstance(results, dict) else results.copy()

    for col in (condition_col, agreement_col, recovery_col):
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in results.")

    summary = (
        df.groupby(condition_col)
          .agg(
              agreement_mean=(agreement_col, "mean"),
              recovery_mean=(recovery_col, "mean"),
              n=("condition", "count"),
          )
          .reset_index()
          .sort_values(condition_col)
    )
    return summary

In [25]:
# kw_acc = kruskal_wallace_with_effect(results, metric="accuracy")
# print("H:", kw_acc["H"], ", df:", kw_acc["df"], ", p:", kw_acc["p"], ", epsion_sq", kw_acc["epsilon_sq"], "\n")
# dunn = kw_acc["dunn_holm_pvals"]  # matrix (or None)

# ar = agreement_recovery_summary(results)
# print(ar)

# Accuracy — 5 conditions
kw_acc = kruskal_wallace_with_effect(results_accuracy, metric="accuracy")
print(f"Accuracy: H({kw_acc['df']}) = {kw_acc['H']:.3f}, "
      f"p = {kw_acc['p']:.4f}, eps2 = {kw_acc['epsilon_sq']:.3f}")
if kw_acc["dunn_holm_pvals"] is not None:
    print(kw_acc["dunn_holm_pvals"], "\n")

# Agreement & Recovery — 4 conditions
for m in ["agreement", "recovery"]:
    kw = kruskal_wallace_with_effect(results_agreement, metric=m)
    print(f"{m.capitalize()}: H({kw['df']}) = {kw['H']:.3f}, "
          f"p = {kw['p']:.4f}, eps2 = {kw['epsilon_sq']:.3f}")
    if kw["dunn_holm_pvals"] is not None:
        print(kw["dunn_holm_pvals"])
    print()

print(agreement_recovery_summary(results_agreement))
print("\nSelective Automation (F) — derived benchmark, not tested:")
print(benchmark_f[["accuracy", "agreement", "recovery"]].mean())

Accuracy: H(4) = 33.948, p = 0.0000, eps2 = 0.315
          A         B         C         D         E
A  1.000000  0.000164  0.000046  0.000462  0.000002
B  0.000164  1.000000  1.000000  1.000000  1.000000
C  0.000046  1.000000  1.000000  1.000000  1.000000
D  0.000462  1.000000  1.000000  1.000000  1.000000
E  0.000002  1.000000  1.000000  1.000000  1.000000 

Agreement: H(3) = 4.527, p = 0.2099, eps2 = 0.020

Recovery: H(3) = 7.587, p = 0.0554, eps2 = 0.060

  condition  agreement_mean  recovery_mean   n
0         B           0.620          0.080  20
1         C           0.720          0.065  20
2         D           0.625          0.100  20
3         E           0.650          0.150  20

Selective Automation (F) — derived benchmark, not tested:
accuracy     0.695
agreement    0.790
recovery     0.125
dtype: float64


### Subjective ratings (Trust)

In [26]:
def analysis_post_hoc_results(users_logs, group_id):
    results = {
        "participant_id": [],
        "condition": [],
        "trust": [],
        "understanding": [],
        "clarity": []
    }

    explanation_map = {"C": "LLM", "D": "OS", "E": ""} 
    explanation = explanation_map.get(group_id, "")

    users_logs_g = users_logs[users_logs["ugroup"] == group_id]
    users = users_logs_g["tuid"].unique()

    def get_single_rating(user_df, block_id):
        s = user_df.loc[user_df["blockId"] == block_id, "faceSelected"]
        if s.empty:
            return np.nan
        return pd.to_numeric(s.iloc[-1], errors="coerce")

    for user in users:
        user_data = users_logs_g[users_logs_g["tuid"] == user]

        block_u = f"block_end_questionnaire_explanation_{explanation}_question_1"
        block_c = f"block_end_questionnaire_explanation_{explanation}_question_2"
        block_t = f"block_end_questionnaire_explanation_{explanation}_question_3"

        understanding = get_single_rating(user_data, block_u)
        clarity       = get_single_rating(user_data, block_c)
        trust         = get_single_rating(user_data, block_t)

        results["participant_id"].append(user)
        results["condition"].append(group_id)
        results["trust"].append(trust)
        results["understanding"].append(understanding)
        results["clarity"].append(clarity)

    return pd.DataFrame(results)


In [27]:
group_c_data = analysis_post_hoc_results(df_raven_2, "C")
group_d_data = analysis_post_hoc_results(df_raven_2, "D")
group_e_data = analysis_post_hoc_results(df_raven_2, "E")

all_data = pd.concat(
    [group_c_data, group_d_data, group_e_data],
    ignore_index=True
)

In [28]:
# all_data

In [29]:
def run_subjective_rating_analysis(results,
                                   metric="trust",
                                   id_col="participant_id",
                                   group_col="condition"):
    # ------------------
    # Kruskal–Wallis
    # ------------------
    conditions = results[group_col].unique()
    groups = [
        results.loc[results[group_col] == c, metric].values
        for c in conditions
    ]

    H, p = kruskal(*groups)

    N = len(results)
    k = results[group_col].nunique()
    epsilon_sq = (H - k + 1) / (N - k)

    print(
        f"{metric.capitalize()} – Kruskal–Wallis: "
        f"H(df={k-1}) = {H:.4f}, p = {p:.6f}, ε² = {epsilon_sq:.4f}"
    )

    # ------------------
    # Dunn post-hoc (Holm)
    # ------------------
    posthoc = None
    if p < 0.05:
        posthoc = sp.posthoc_dunn(
            results,
            val_col=metric,
            group_col=group_col,
            p_adjust="holm"
        )
        print("\nDunn post-hoc (Holm-corrected p-values):")
        print(posthoc)

    # ------------------
    # Cliff’s delta
    # ------------------
    def cliffs_delta(a, b):
        a = np.asarray(a)
        b = np.asarray(b)
        gt = sum(x > y for x in a for y in b)
        lt = sum(x < y for x in a for y in b)
        return (gt - lt) / (len(a) * len(b))

    deltas = {}
    conds = list(conditions)

    print("\nCliff’s δ:")
    for i in range(len(conds)):
        for j in range(i + 1, len(conds)):
            c1, c2 = conds[i], conds[j]
            a = results.loc[results[group_col] == c1, metric].values
            b = results.loc[results[group_col] == c2, metric].values
            d = cliffs_delta(a, b)
            deltas[(c1, c2)] = d
            print(f"  {c1} vs {c2}: δ = {d:.3f}")

    # ------------------
    # Return structured output
    # ------------------
    return {
        "kruskal": {
            "H": H,
            "df": k - 1,
            "p": p,
            "epsilon_sq": epsilon_sq
        },
        "posthoc_dunn": posthoc,
        "cliffs_delta": deltas
    }

In [30]:
summary_trust = run_subjective_rating_analysis(all_data, metric="trust")
summary_understanding = run_subjective_rating_analysis(all_data, metric="understanding")
summary_clarity = run_subjective_rating_analysis(all_data, metric="clarity")

Trust – Kruskal–Wallis: H(df=2) = 5.1799, p = 0.075025, ε² = 0.0558

Cliff’s δ:
  C vs D: δ = 0.223
  C vs E: δ = -0.168
  D vs E: δ = -0.393
Understanding – Kruskal–Wallis: H(df=2) = 3.1116, p = 0.211016, ε² = 0.0195

Cliff’s δ:
  C vs D: δ = 0.323
  C vs E: δ = 0.095
  D vs E: δ = -0.182
Clarity – Kruskal–Wallis: H(df=2) = 7.5320, p = 0.023144, ε² = 0.0971

Dunn post-hoc (Holm-corrected p-values):
          C         D         E
C  1.000000  0.730391  0.057682
D  0.730391  1.000000  0.034194
E  0.057682  0.034194  1.000000

Cliff’s δ:
  C vs D: δ = 0.060
  C vs E: δ = -0.385
  D vs E: δ = -0.448


# Study 3: LSAT between subjects

## Group Accuracy Comparisons

In [32]:
# Load the dataset
main = "study_3_lsat_between_subjects"
df_lsat = pd.read_csv(f'{main}/results/questionnaires_answers_log.csv')

In [33]:
def analysis_test_results(users_logs, group_id):
    results = {
        "participant_id": [],
        "condition": [],
        "accuracy": [],
        "agreement": [],
        "recovery": []
    }
    
    users_logs = users_logs[users_logs["ugroup"] == group_id]
    users = users_logs['tuid'].unique()
    
    for user in users:
        user_data = users_logs[users_logs["tuid"] == user]

        correct_count = 0
        questions = 0
        agreement = 0
        recovery = 0 
        
        for _, row in user_data.iterrows():
            blockId = row['selectedOptionName']
            true_choice = row['correctAnswerValue']
            user_choice = row['selectedOptionValue']
            pred_choice = row['aiSelectedOptionValue']

            # check if it is feedback or answer
            if "facesSlider" not in blockId: # ---> answer
                questions  += 1
                
                # check how many qeusiotns did the user answered correctly
                if user_choice == true_choice:
                    correct_count += 1

                agreement += (1 if user_choice == pred_choice else 0)
                recovery += 1 if (
                    pred_choice != true_choice and
                    user_choice == true_choice
                ) else 0
        
        if questions == 10: 
            results["participant_id"].append(user)
            results["condition"].append(group_id)
            results["accuracy"].append(correct_count/questions)
            results["agreement"].append(agreement/questions)
            results["recovery"].append(recovery/questions)
        
    return pd.DataFrame(results)

In [34]:
group_a_data = analysis_test_results(df_lsat, "A").head(20)
group_b_data = analysis_test_results(df_lsat, "B").head(20)
group_c_data = analysis_test_results(df_lsat, "C").head(20)
group_d_data = analysis_test_results(df_lsat, "D").head(20)

results = pd.concat(
    [group_a_data, group_b_data, group_c_data, group_d_data],
    ignore_index=True
)

In [35]:
results

,participant_id,condition,accuracy,agreement,recovery
0,UID_OK52uh59yc17Gg38pe,A,0.3,0.3,0.1
1,UID_Op14Zg81Cu60eo02dG,A,0.6,0.9,0.0
2,UID_ab60IH48Gb39WA55eu,A,0.6,1.0,0.0
3,UID_tS40Mq77cQ54TK32dV,A,0.6,0.8,0.1
4,UID_lL50wb46bB48Jr80GT,A,0.7,0.9,0.1
...,...,...,...,...,...
75,UID_wh63At64VX53cv82wk,D,0.4,0.6,0.1
76,UID_we29LM61Gb71sd27Bc,D,0.6,0.9,0.0
77,UID_QQ92Ft68nh86oU22jZ,D,0.5,0.4,0.2
78,UID_dT19xs25DD43WR91Ct,D,0.2,0.5,0.0


In [36]:
summary = run_group_accuracy_tests(results)

Kruskal–Wallis: H = 22.2970, df = 3, p = 0.000057, ε² = 0.2539

Dunn post-hoc (Holm-corrected p-values):
          A         B         C         D
A  1.000000  0.000569  0.736336  0.736336
B  0.000569  1.000000  0.012333  0.000117
C  0.736336  0.012333  1.000000  0.568771
D  0.736336  0.000117  0.568771  1.000000
Cliff's δ (A vs B) = -0.7000
Cliff's δ (A vs C) = -0.1650
Cliff's δ (A vs D) = 0.0800
Cliff's δ (B vs C) = 0.5625
Cliff's δ (B vs D) = 0.7400
Cliff's δ (C vs D) = 0.2625


### Agreement & Recovery tests

In [37]:
kw_acc = kruskal_wallace_with_effect(results, metric="accuracy")
print("H:", kw_acc["H"], ", df:", kw_acc["df"], ", p:", kw_acc["p"], ", epsion_sq", kw_acc["epsilon_sq"], "\n")
dunn = kw_acc["dunn_holm_pvals"] 

ar = agreement_recovery_summary(results)
print(ar)

H: 22.29696651704475 , df: 3 , p: 5.657806485879269e-05 , epsion_sq 0.25390745417164146 

  condition  agreement_mean  recovery_mean   n
0         A           0.600          0.110  20
1         B           0.740          0.190  20
2         C           0.620          0.145  20
3         D           0.525          0.140  20


## Subjective ratings (Trust)

In [38]:
def analysis_lsat_post_hoc_results(users_logs, group_id):
    results = {
        "participant_id": [],
        "condition": [],
        "trust": [],
        "understanding": [],
        "clarity": []
    }

    explanation_map = {
        "A": "ONLY_PREDICTION",
        "B": "LLM_EXPLANATION",
        "C": "OTHER_EXPLANATION",
        "D": "PREDICTION_WITH_CONFIDENCE",
    }
    explanation = explanation_map.get(group_id, "")

    users_logs_g = users_logs[users_logs["ugroup"] == group_id]
    users = users_logs_g["tuid"].unique()

    def get_single_rating(user_df, block_id):
        s = user_df.loc[user_df["selectedOptionName"] == block_id, "selectedOptionValue"]
        if s.empty:
            return np.nan
        return pd.to_numeric(s.iloc[-1], errors="coerce")

    for user in users:
        user_data = users_logs_g[users_logs_g["tuid"] == user]

        block_u = f"facesSlider_block_end_questionnaire_explanation_{explanation}_question_1"
        block_c = f"facesSlider_block_end_questionnaire_explanation_{explanation}_question_2"
        block_t = f"facesSlider_block_end_questionnaire_explanation_{explanation}_question_3"

        understanding = get_single_rating(user_data, block_u)
        clarity       = get_single_rating(user_data, block_c)
        trust         = get_single_rating(user_data, block_t)

        if pd.notna(understanding) and pd.notna(clarity) and pd.notna(trust):
            results["participant_id"].append(user)
            results["condition"].append(group_id)
            results["trust"].append(trust)
            results["understanding"].append(understanding)
            results["clarity"].append(clarity)

    return pd.DataFrame(results)

In [39]:
group_a_data = analysis_lsat_post_hoc_results(df_lsat, "A").head(20)
group_b_data = analysis_lsat_post_hoc_results(df_lsat, "B").head(20)
group_c_data = analysis_lsat_post_hoc_results(df_lsat, "C").head(20)
group_d_data = analysis_lsat_post_hoc_results(df_lsat, "D").head(20)

all_data = pd.concat(
    [group_a_data, group_b_data, group_c_data, group_d_data],
    ignore_index=True
)

In [40]:
all_data

,participant_id,condition,trust,understanding,clarity
0,UID_OK52uh59yc17Gg38pe,A,2,2,2
1,UID_Op14Zg81Cu60eo02dG,A,4,2,3
2,UID_ab60IH48Gb39WA55eu,A,5,5,3
3,UID_tS40Mq77cQ54TK32dV,A,4,5,4
4,UID_lL50wb46bB48Jr80GT,A,3,4,3
...,...,...,...,...,...
75,UID_wh63At64VX53cv82wk,D,5,5,5
76,UID_we29LM61Gb71sd27Bc,D,4,3,4
77,UID_QQ92Ft68nh86oU22jZ,D,4,3,3
78,UID_dT19xs25DD43WR91Ct,D,3,3,4


In [41]:
summary_trust = run_subjective_rating_analysis(all_data, metric="trust")
summary_understanding = run_subjective_rating_analysis(all_data, metric="understanding")
summary_clarity = run_subjective_rating_analysis(all_data, metric="clarity")

Trust – Kruskal–Wallis: H(df=3) = 5.4203, p = 0.143482, ε² = 0.0318

Cliff’s δ:
  A vs B: δ = -0.300
  A vs C: δ = 0.030
  A vs D: δ = -0.090
  B vs C: δ = 0.395
  B vs D: δ = 0.225
  C vs D: δ = -0.150
Understanding – Kruskal–Wallis: H(df=3) = 11.3238, p = 0.010098, ε² = 0.1095

Dunn post-hoc (Holm-corrected p-values):
          A         B         C         D
A  1.000000  0.509415  0.551835  0.245430
B  0.509415  1.000000  0.245430  0.005558
C  0.551835  0.245430  1.000000  0.509415
D  0.245430  0.005558  0.509415  1.000000

Cliff’s δ:
  A vs B: δ = -0.225
  A vs C: δ = 0.110
  A vs D: δ = 0.318
  B vs C: δ = 0.375
  B vs D: δ = 0.560
  C vs D: δ = 0.273
Clarity – Kruskal–Wallis: H(df=3) = 10.7649, p = 0.013068, ε² = 0.1022

Dunn post-hoc (Holm-corrected p-values):
          A         B         C         D
A  1.000000  0.011899  1.000000  0.613912
B  0.011899  1.000000  0.063131  0.272434
C  1.000000  0.063131  1.000000  1.000000
D  0.613912  0.272434  1.000000  1.000000

Cliff’s δ:
